# FINAL THORACIC HUMAN IN LOOP TEST

In [68]:
!pip install -U torchao --break-system-packages -q

In [69]:
!pip install trl transformers accelerate peft datasets bitsandbytes

In [70]:
"""
Sentence-level HITL memory generalization + self-consistency eval.

Run on Kaggle with a GPU. Set MODALITY below, then run top to bottom.

Two tests, matching what you described:

  TEST 1 - Pair generalization (20 cases):
      Find 20 pairs of DIFFERENT reports that each contain a near-duplicate
      incidental-finding sentence. Modify report A's version of that sentence
      and store it as a correction. Then run report B (unmodified) through
      the pipeline and check:
        (1) the original/unmodified sentence is NOT in the final output
        (2) the modification IS in the final output
      This tests whether a correction on one report's finding generalizes to
      a paraphrased/near-duplicate finding in a different report.

  TEST 2 - Self-consistency (10 cases):
      Take 10 standalone reports (not used in the pairs above). Modify one of
      their gold incidental sentences, store it as a correction, then resend
      the EXACT SAME report and check the corrected version -- not the old
      one -- is what comes back out.

Memory is keyed on individual incidental sentences (not whole reports) --
see the chat message before this file for why.
"""

import gc
import json
import re
from dataclasses import dataclass, field
from typing import Callable, Optional
import os

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [71]:
# ===========================================================================
# 0. CONFIG -- edit this block per modality
# ===========================================================================

MODALITY = "thoracic"  # "thoracic" | "abdomen"

PATHS = {
    "abdomen": dict(
        kd_base="/kaggle/input/datasets/mythreyeehari20/kd-abdomen-weights/kd_merged_abdomen",
        qlora_adapter="/kaggle/input/datasets/mythreyeehari20/abdomen-kd-qlora-best-r32-alpha64/qlora_best_r32_alpha64",
        merged_dir="/kaggle/working/kd_qlora_merged_abdomen",
        train_jsonl="/kaggle/input/datasets/mythreyee1006/train-dataset-abdomen-new/train_records_abdomenCT.jsonl",
        test_jsonl="/kaggle/input/datasets/mythreyee1006/test-dataset-abdomen-final/test_records_abdomenCT.jsonl",
    ),
    "thoracic": dict(
        kd_base="/kaggle/input/datasets/mythreyeeh/kd-best-thoracic",
        qlora_adapter="/kaggle/input/datasets/mythreyeeh/qlora-weights-after-kd/qlora_best_r32_alpha64",
        merged_dir="/kaggle/working/kd_qlora_merged_thoracic",
        test_unannotated="/kaggle/input/datasets/mythreyeeh/test-dataset-thoractic/Test_dataset_thoractic/test_dataset_thoracic_unannotated.json",
        test_annotated="/kaggle/input/datasets/mythreyeeh/test-dataset-thoractic/Test_dataset_thoractic/test_dataset_thoracic_annotated.json",
        train_unannotated="/kaggle/input/datasets/mythreyeeh/train-dataset-thoractic/Train_dataset_thoractic/dataset_thoracic_unannotated.json",
        train_annotated="/kaggle/input/datasets/mythreyeeh/train-dataset-thoractic/Train_dataset_thoractic/dataset_thoracic_annotated.json",
    ),
}

SIM_THRESHOLD = 0.9  # tune empirically -- see note in earlier message
N_PAIRS = 100
N_STANDALONE = 100

In [72]:
# ===========================================================================
# 1. Dataset loaders -- normalize both formats to a common schema:
#    {report_id, free_text, gold: {contains_IF, incidental_sentences}}
# ===========================================================================

def load_abdomen_records(jsonl_path: str) -> list[dict]:
    records = []
    with open(jsonl_path) as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            msgs = d["messages"]
            user_msg = next(m["content"] for m in msgs if m["role"] == "user")
            assistant_msg = next(m["content"] for m in msgs if m["role"] == "assistant")
            free_text = user_msg.split("Report:\n", 1)[-1].strip()
            gold = json.loads(assistant_msg)
            report_id = d.get("report_id", f"abd_{i:05d}")  # not present in sample -- synthesized
            records.append({
                "report_id": report_id,
                "free_text": free_text,
                "gold": {
                    "contains_IF": gold["contains_IF"],
                    "incidental_sentences": gold["incidental_sentences"],
                },
            })
    return records


def load_thoracic_records(unannotated_path: str, annotated_path: str) -> list[dict]:
    with open(unannotated_path, encoding="utf-8") as f:
        unannotated = json.load(f)["reports"]
    with open(annotated_path, encoding="utf-8") as f:
        annotated = json.load(f)["reports"]
    lookup = {r["report_id"]: r["annotation"] for r in annotated}

    records = []
    for report in unannotated:
        rid = report["report_id"]
        if rid in lookup:
            ann = lookup[rid]
            records.append({
                "report_id": rid,
                "free_text": report["free_text"],
                "gold": {
                    "contains_IF": ann["contains_IF"],
                    "incidental_sentences": ann["incidental_sentences"],
                },
            })
    return records


def load_records_for_modality(modality: str, split: str = "train") -> list[dict]:
    p = PATHS[modality]
    if modality == "abdomen":
        return load_abdomen_records(p["train_jsonl" if split == "train" else "test_jsonl"])
    else:
        if split == "train":
            return load_thoracic_records(p["train_unannotated"], p["train_annotated"])
        return load_thoracic_records(p["test_unannotated"], p["test_annotated"])

In [73]:
# ===========================================================================
# 2. Model loading -- same merge pattern you used, parameterized
# ===========================================================================

def load_merged_model(kd_base_path: str, qlora_adapter_path: str, merged_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(kd_base_path, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_for_merge = AutoModelForCausalLM.from_pretrained(
        kd_base_path, torch_dtype=torch.bfloat16, device_map={"": 0}, trust_remote_code=True,
    )
    merge_model = PeftModel.from_pretrained(base_for_merge, qlora_adapter_path)
    merged_model = merge_model.merge_and_unload()

    merged_model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)

    del base_for_merge, merge_model
    gc.collect()
    torch.cuda.empty_cache()

    return merged_model, tokenizer


# --- Use YOUR actual build_messages / build_system_prompt / parse_output from your
# training pipeline instead of anything defined here. Paste those three functions
# (and EVAL_BATCH_SIZE if you want to reuse generate_predictions too) into this
# notebook before this cell, then this script calls them directly by name.
#
# extract_fn below is kept ONLY as a single-report fallback for ad-hoc checks --
# for the actual test harness, use `run_batched_raw_predictions` further down,
# which wraps your own `generate_predictions` for proper batched GPU inference.

def build_extract_fn(model, tokenizer, max_new_tokens: int = 256) -> Callable[[str], dict]:
    """Single-report fallback. Prefer run_batched_raw_predictions for the actual eval."""
    device = next(model.parameters()).device

    def extract_fn(free_text: str) -> dict:
        record = {"free_text": free_text, "report_id": "_adhoc_", "gold": {"incidental_sentences": []}}
        messages = build_messages(record)  # <-- your function, must be in notebook scope
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = out[0][enc["input_ids"].shape[1]:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        parsed = parse_output(text)  # <-- your function
        return parsed if parsed is not None else {"contains_IF": False, "incidental_sentences": []}

    return extract_fn


def run_batched_raw_predictions(model, tokenizer, records: list[dict]) -> dict:
    """
    Thin wrapper around YOUR generate_predictions -- batches inference for all
    records at once instead of one .generate() call per report. Returns a dict
    keyed by report_id -> list of predicted incidental sentences (raw, before
    any memory override).
    """
    raw_results = generate_predictions(model, tokenizer, records)  # <-- your function
    return {r["report_id"]: (r["pred_sentences"] or []) for r in raw_results}


def sentence_in_list(target: str, sentence_list: list[str], threshold: float = 0.85) -> bool:
    """Fuzzy membership check using your SequenceMatcher-based similarity()."""
    t = target.strip().lower()
    return any(similarity(t, s.strip().lower()) >= threshold for s in sentence_list)


def exact_in_list(target: str, sentence_list: list[str]) -> bool:
    """
    Exact (case/whitespace-normalized) membership check. Use this -- not
    sentence_in_list -- for "did the untouched original leak through" checks.
    A modified sentence that appends text (e.g. "...arteries, new since prior
    exam.") fuzzy-matches its own unmodified prefix at >=0.85, which falsely
    flags the original as still present when only the correction is there.
    """
    t = target.strip().lower()
    return any(t == s.strip().lower() for s in sentence_list)

In [74]:
# ===========================================================================
# 3. Embedding backend (sentence-level -- BERT-style is fine, fast for short spans)
# ===========================================================================

def normalize_sentence(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'(\d+(\.\d+)?)\s*(mm|cm)', r'\1\3', s)
    return s

class BertStyleEmbedding:
    def __init__(self, model_name: str = "emilyalsentzer/Bio_ClinicalBERT", device: str = "cuda"):
        from transformers import AutoTokenizer as _AT, AutoModel as _AM
        self.device = device if torch.cuda.is_available() else "cpu"
        self.tokenizer = _AT.from_pretrained(model_name)
        self.model = _AM.from_pretrained(model_name).to(self.device).eval()
        self.dim = self.model.config.hidden_size

    def embed_batch(self, texts: list[str]) -> np.ndarray:
        texts = [normalize_sentence(t) for t in texts]
        with torch.no_grad():
            enc = self.tokenizer(texts, padding=True, truncation=True, max_length=128,
                                  return_tensors="pt").to(self.device)
            out = self.model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            pooled = torch.nn.functional.normalize(pooled, dim=-1)
        return pooled.cpu().numpy().astype("float32")

    def embed_one(self, text: str) -> np.ndarray:
        return self.embed_batch([text])[0]

In [75]:
# ===========================================================================
# 4. Sentence-level memory
# ===========================================================================

@dataclass
class SentenceMemory:
    backend: BertStyleEmbedding
    sim_threshold: float = SIM_THRESHOLD
    trigger_sentences: list = field(default_factory=list)
    corrected: list = field(default_factory=list)
    is_if_flags: list = field(default_factory=list)
    embs: np.ndarray = field(default_factory=lambda: np.zeros((0, 1), dtype="float32"))

    def __post_init__(self):
        self.embs = np.zeros((0, self.backend.dim), dtype="float32")

    def add_correction(self, trigger_sentence: str, corrected_sentence: Optional[str], is_if: bool):
        v = self.backend.embed_one(trigger_sentence)[None, :]
        self.embs = np.vstack([self.embs, v]) if self.embs.shape[0] else v
        self.trigger_sentences.append(trigger_sentence)
        self.corrected.append(corrected_sentence)
        self.is_if_flags.append(is_if)

    def lookup(self, candidate_sentence: str) -> Optional[dict]:
        if self.embs.shape[0] == 0:
            return None
        v = self.backend.embed_one(candidate_sentence)
        sims = self.embs @ v
        idx = int(np.argmax(sims))
        sim = float(sims[idx])
        char_sim = max(similarity(candidate_sentence, t) for t in self.trigger_sentences)
        if sim >= self.sim_threshold or char_sim >= 0.90:
            return {
                "corrected_sentence": self.corrected[idx],
                "is_IF": self.is_if_flags[idx],
                "similarity": max(sim, char_sim),
                "matched_trigger": self.trigger_sentences[idx],
            }
        return None


def apply_memory_override(raw_output: dict, memory: SentenceMemory) -> dict:
    """Post-process an LLM extraction, overriding any sentence that matches memory."""
    final_sentences = []
    for s in raw_output.get("incidental_sentences", []):
        hit = memory.lookup(s)
        if hit is not None:
            if hit["is_IF"] and hit["corrected_sentence"]:
                final_sentences.append(hit["corrected_sentence"])
            # else: memory says this should be excluded -- drop it
        else:
            final_sentences.append(s)
    return {"contains_IF": len(final_sentences) > 0, "incidental_sentences": final_sentences}

In [76]:
# ===========================================================================
# 5. Modification function -- REPLACE with real radiologist-style edits when you can.
#    This heuristic version exists so the harness is runnable standalone.
# ===========================================================================

def default_modify_sentence(sentence: str) -> str:
    m = re.search(r"(\d+(\.\d+)?)\s?(mm|cm)", sentence)
    if m:
        val = float(m.group(1)) + (2 if m.group(3) == "mm" else 0.5)
        val_str = str(int(val)) if val == int(val) else str(val)
        return sentence[:m.start()] + f"{val_str} {m.group(3)}" + sentence[m.end():]
    if re.search(r"\bstable\b|\bunchanged\b", sentence, re.I):
        return re.sub(r"\bstable\b|\bunchanged\b", "increased in size", sentence, flags=re.I)
    return sentence.rstrip(".") + ", new since prior exam."

In [77]:
# ===========================================================================
# 6. Pair + standalone selection
# ===========================================================================

def build_pairs_and_standalone(records, backend,
                                           n_pairs=N_PAIRS, n_standalone=N_STANDALONE,
                                           min_len=15,
                                           sim_bins=((0.98, 1.001), (0.95, 0.98), (0.90, 0.95))):
    by_id = {r["report_id"]: r for r in records}
    pool = []
    for r in records:
        if r["gold"]["contains_IF"]:
            for s in r["gold"]["incidental_sentences"]:
                if len(s) >= min_len:
                    pool.append((r["report_id"], s))

    if len(pool) < 2:
        raise ValueError("Not enough incidental sentences to build pairs.")

    texts = [s for _, s in pool]
    embs = backend.embed_batch(texts)
    sims = embs @ embs.T
    n = len(pool)
    np.fill_diagonal(sims, -1)

    report_ids = np.array([rid for rid, _ in pool])
    same_report = report_ids[:, None] == report_ids[None, :]
    sims[same_report] = -1

    iu = np.triu_indices(n, k=1)
    all_sims = sims[iu]

    used_reports = set()
    per_bin_target = max(1, n_pairs // len(sim_bins))
    pairs = []

    for lo, hi in sim_bins:
        in_band = [k for k in range(len(all_sims)) if lo <= all_sims[k] < hi]
        in_band.sort(key=lambda k: -all_sims[k])
        count_this_bin = 0
        for k in in_band:
            i, j = iu[0][k], iu[1][k]
            rid_a, sent_a = pool[i]
            rid_b, sent_b = pool[j]
            if rid_a in used_reports or rid_b in used_reports:
                continue
            pairs.append({
                "report_a": rid_a, "sentence_a": sent_a,
                "report_b": rid_b, "sentence_b": sent_b,
                "similarity": float(all_sims[k]),
                "report_b_free_text": by_id[rid_b]["free_text"],
            })
            used_reports.update([rid_a, rid_b])
            count_this_bin += 1
            if count_this_bin >= per_bin_target:
                break
        print(f"  bin {lo:.2f}-{min(hi,1.0):.2f}: found {count_this_bin} pairs (target {per_bin_target})")

    standalone = [
        r for r in records
        if r["report_id"] not in used_reports
        and r["gold"]["contains_IF"] and r["gold"]["incidental_sentences"]
    ][:n_standalone]

    return pairs, standalone

In [78]:
# ===========================================================================
# 7. The two test harnesses
# ===========================================================================

def run_pair_generalization_test(pairs: list[dict], raw_preds: dict,
                                  embed_backend: BertStyleEmbedding,
                                  modify_fn: Callable[[str], str] = default_modify_sentence,
                                  fuzzy_threshold: float = 0.85) -> list[dict]:
    """raw_preds: report_id -> list[str], from run_batched_raw_predictions."""
    results = []
    for pair in pairs:
        memory = SentenceMemory(backend=embed_backend)  # fresh memory per case -- isolates the test
        modified = modify_fn(pair["sentence_a"])
        memory.add_correction(trigger_sentence=pair["sentence_a"], corrected_sentence=modified, is_if=True)

        raw_sentences_b = raw_preds.get(pair["report_b"], [])
        final_output_b = apply_memory_override(
            {"incidental_sentences": raw_sentences_b}, memory)
        final_sents = final_output_b["incidental_sentences"]

        suppressed_original = (
            not exact_in_list(pair["sentence_b"], final_sents)
            and not exact_in_list(pair["sentence_a"], final_sents)
        )
        injected_modification = sentence_in_list(modified, final_sents, fuzzy_threshold)

        results.append({
            "report_b": pair["report_b"],
            "similarity_a_b": pair["similarity"],
            "sentence_a": pair["sentence_a"],
            "sentence_b": pair["sentence_b"],
            "modification": modified,
            "raw_llm_sentences": raw_sentences_b,
            "final_sentences": final_sents,
            "suppressed_original": suppressed_original,
            "injected_modification": injected_modification,
            "pass": suppressed_original and injected_modification,
        })
    return results


def run_self_consistency_test(standalone: list[dict], raw_preds: dict,
                               embed_backend: BertStyleEmbedding,
                               modify_fn: Callable[[str], str] = default_modify_sentence,
                               fuzzy_threshold: float = 0.85) -> list[dict]:
    results = []
    for rec in standalone:
        raw_sentences = raw_preds.get(rec["report_id"], [])
        if not raw_sentences:
            # nothing the model flagged the first time -- extraction miss, not a memory test case
            results.append({
                "report_id": rec["report_id"], "original_sentence": None, "modification": None,
                "raw_llm_sentences": raw_sentences, "final_sentences": [],
                "returns_corrected": False, "still_has_original": False, "pass": False,
            })
            continue

        memory = SentenceMemory(backend=embed_backend)
        original = raw_sentences[0]  # trigger off the model's OWN first-pass wording, not gold
        modified = modify_fn(original)
        memory.add_correction(trigger_sentence=original, corrected_sentence=modified, is_if=True)

        final_output = apply_memory_override({"incidental_sentences": raw_sentences}, memory)
        final_sents = final_output["incidental_sentences"]

        returns_corrected = sentence_in_list(modified, final_sents, fuzzy_threshold)
        still_has_original = exact_in_list(original, final_sents)

        results.append({
            "report_id": rec["report_id"],
            "original_sentence": original,
            "modification": modified,
            "raw_llm_sentences": raw_sentences,
            "final_sentences": final_sents,
            "returns_corrected": returns_corrected,
            "still_has_original": still_has_original,
            "pass": returns_corrected and not still_has_original,
        })
    return results


def summarize(results: list[dict], label: str) -> dict:
    n = len(results)
    passed = sum(r["pass"] for r in results)
    extraction_miss = sum(1 for r in results if not r["pass"] and not r["raw_llm_sentences"])
    memory_mismatch = sum(1 for r in results if not r["pass"] and r["raw_llm_sentences"])
    print(f"\n=== {label} ===")
    print(f"n_cases: {n}   passed: {passed}   pass_rate: {passed / n:.2%}" if n else "no cases")
    print(f"failures -- extraction_miss: {extraction_miss}   memory_mismatch: {memory_mismatch}")
    return {"n_cases": n, "n_passed": passed, "pass_rate": passed / n if n else None,
            "extraction_miss": extraction_miss, "memory_mismatch": memory_mismatch}

In [79]:
#===========================================================================
# IMPORT OLD FUNCTIONS
#===========================================================================
# ===========================================================================
# Modality-aware build_system_prompt + shared eval helpers
# Paste this BEFORE the main script cell. Assumes MODALITY is already set
# ("abdomen" or "thoracic") from your CONFIG block.
# ===========================================================================

import re, json, gc
from difflib import SequenceMatcher

import torch
import wandb
from transformers import TrainerCallback, TrainerState, TrainerControl

EVAL_BATCH_SIZE = 8
EVAL_STEPS = 50
FOCAL_GAMMA = 2.0

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key)
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not set -- running with wandb disabled.")


# --- ACR retrieval + build_system_prompt, branched by modality ---

if MODALITY == "abdomen":
    ACR_GUIDELINES_PATH = "/kaggle/input/datasets/mythreyeehari20/acr-rules-abdomen/Findings_Extracted_WhitePapers.json"

    with open(ACR_GUIDELINES_PATH) as f:
        acr_guidelines = json.load(f)

    def condense_finding(finding):
        feat_str = "; ".join(finding["features"])
        return f"[{finding['finding_id']}] {finding['finding_name']} \u2014 {feat_str}"

    def build_condensed_index(guidelines):
        # abdomen guidelines are flat: {"findings": [...]}
        index = []
        for finding in guidelines["findings"]:
            searchable = " ".join([finding["finding_name"], " ".join(finding["features"])]).lower()
            index.append({
                "finding_id": finding["finding_id"],
                "searchable_text": searchable,
                "condensed_line": condense_finding(finding),
            })
        return index

    TASK_INSTRUCTIONS_TEMPLATE = (
        "You are a clinical assistant specialized in abdominopelvic radiology. Your task is to identify "
        "INCIDENTAL findings in a free-text abdominopelvic CT report \u2014 findings unrelated to the report's "
        "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
        "Use the reference guidelines below to judge whether a finding is clinically incidental "
        "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
        "expected finding tied to the report's main indication.\n\n"
        "Extract the EXACT sentence(s) from the report that describe incidental findings \u2014 do not "
        "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
        "REFERENCE GUIDELINES:\n{acr_context}\n\n"
        "OUTPUT FORMAT:\n"
        "Return ONLY one JSON object and nothing else.\n"
        "The JSON object MUST contain exactly these two fields: "
        "contains_IF and incidental_sentences.\n"
        "contains_IF MUST be a boolean: true or false.\n"
        "incidental_sentences MUST be a JSON array of STRINGS, not objects.\n"
        "Each string must be an EXACT sentence copied from the report.\n"
        "Do NOT add fields such as sentence, if, is_incidental, findings, "
        "reference_guidelines, or any other fields.\n"
        "If there are no incidental findings, use an empty array and set contains_IF to false.\n"
        "If incidental findings are present, set contains_IF to true and include only "
        "the exact sentences containing those findings.\n"
        "Required format:\n"
        "{{\"contains_IF\": false, \"incidental_sentences\": []}}\n"
    )

elif MODALITY == "thoracic":
    ACR_GUIDELINES_PATH = "/kaggle/input/datasets/mythreyeeh/white-paper-guidelines/acr_thoracic_incidental_findings.json"

    with open(ACR_GUIDELINES_PATH) as f:
        acr_guidelines = json.load(f)

    def condense_finding(finding):
        feat_str = "; ".join(finding["features"])
        return f"[{finding['finding_id']}] {finding['finding_name']} \u2014 {feat_str}"

    def build_condensed_index(guidelines):
        # thoracic guidelines are nested: [{"organ_system": ..., "findings": [...]}, ...]
        index = []
        for organ_system in guidelines:
            for finding in organ_system["findings"]:
                searchable = " ".join([finding["finding_name"], " ".join(finding["features"])]).lower()
                index.append({
                    "finding_id": finding["finding_id"],
                    "organ_system": organ_system["organ_system"],
                    "searchable_text": searchable,
                    "condensed_line": condense_finding(finding),
                })
        return index

    TASK_INSTRUCTIONS_TEMPLATE = (
        "You are a clinical assistant specialized in thoracic radiology. Your task is to identify "
        "INCIDENTAL findings in a free-text thoracic CT report \u2014 findings unrelated to the report's "
        "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
        "Use the reference guidelines below to judge whether a finding is clinically incidental "
        "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
        "expected finding tied to the report's main indication.\n\n"
        "Extract the EXACT sentence(s) from the report that describe incidental findings \u2014 do not "
        "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
        "REFERENCE GUIDELINES:\n{acr_context}"
    )

else:
    raise ValueError(f"Unknown MODALITY: {MODALITY!r} -- expected 'abdomen' or 'thoracic'")


ACR_INDEX = build_condensed_index(acr_guidelines)
full_condensed = "\n".join(f["condensed_line"] for f in ACR_INDEX)
print(f"Indexed {len(ACR_INDEX)} ACR findings for modality={MODALITY} ({len(full_condensed):,} chars condensed)")

STOPWORDS = {"the","a","an","of","in","on","to","and","or","with","is","are","was","were",
             "at","for","by","as","be","no","not","also","this","that","been","has","have"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return set(w for w in words if w not in STOPWORDS and len(w) > 2)

def retrieve_relevant_findings(report_text, index, top_k=20, min_overlap=1):
    report_tokens = tokenize(report_text)
    scored = []
    for entry in index:
        finding_tokens = tokenize(entry["searchable_text"])
        overlap = len(report_tokens & finding_tokens)
        if overlap >= min_overlap:
            scored.append((overlap, entry))
    scored.sort(key=lambda x: -x[0])
    return [entry for _, entry in scored[:top_k]]

def build_filtered_acr_context(report_text, index, top_k=10):
    relevant = retrieve_relevant_findings(report_text, index, top_k=top_k)
    if not relevant:
        return full_condensed
    return "\n".join(e["condensed_line"] for e in relevant)

def build_system_prompt(report_text, top_k=10):
    filtered_context = build_filtered_acr_context(report_text, ACR_INDEX, top_k=top_k)
    return TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)


# --- Shared eval helpers (identical in both source notebooks) ---

def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None


def build_messages(record, few_shot_pool=None, n_shot=0):
    messages = [{"role": "system", "content": build_system_prompt(record["free_text"])}]
    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            messages.append({"role": "user", "content": f"Report:\n{ex['free_text']}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })
    messages.append({"role": "user", "content": f"Report:\n{record['free_text']}"})
    return messages


def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def fuzzy_match_sets(gold_set, pred_set, threshold=0.85):
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    matched_gold, matched_pred = set(), set()
    pairs = []
    for gi, g in enumerate(gold_list):
        for pi, p in enumerate(pred_list):
            sim = similarity(g, p)
            if sim >= threshold:
                pairs.append((sim, gi, pi))
    pairs.sort(key=lambda x: -x[0])
    for sim, gi, pi in pairs:
        if gi in matched_gold or pi in matched_pred:
            continue
        matched_gold.add(gi)
        matched_pred.add(pi)
    tp = len(matched_gold)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def run_evaluation_fuzzy(results, label="", threshold=0.85):
    parse_failures = sum(r["parse_failed"] for r in results)
    valid = [r for r in results if not r["parse_failed"]]
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []
    for r in valid:
        gold_set = set(s.strip().lower() for s in r["gold_sentences"])
        pred_set = set(s.strip().lower() for s in (r["pred_sentences"] or []))
        if len(gold_set) == 0:
            neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
        else:
            tp, fp, fn = fuzzy_match_sets(gold_set, pred_set, threshold=threshold)
            total_tp += tp; total_fp += fp; total_fn += fn
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            pos_scores.append(f1)
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg > 0 and n_pos > 0:
        macro_f1 = (avg_neg + avg_pos) / 2
    elif n_neg > 0:
        macro_f1 = avg_neg
    elif n_pos > 0:
        macro_f1 = avg_pos
    else:
        macro_f1 = 0.0
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                if (micro_precision + micro_recall) > 0 else 0.0)
    print(f"\n{'='*50}\n=== {label} (fuzzy threshold={threshold}) ===\n{'='*50}")
    print(f"Parse failures:      {parse_failures}/{len(results)}")
    print(f"Negative-report acc: {avg_neg:.4f}  (n={n_neg})")
    print(f"Positive-report F1:  {avg_pos:.4f}  (n={n_pos})")
    print(f"Sentence Macro F1:   {macro_f1:.4f}")
    print(f"Sentence Weighted:   {weighted_f1:.4f}")
    print(f"Sentence Micro F1:   {micro_f1:.4f}")
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1,
            "parse_failures": parse_failures, "n_neg": n_neg, "n_pos": n_pos,
            "avg_neg": avg_neg, "avg_pos": avg_pos}


def generate_predictions(model, tokenizer, records, few_shot_pool=None, n_shot=0,
                          n=None, batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    results = []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)

            results.append({
                "report_id": record["report_id"],
                "gold_sentences": record["gold"]["incidental_sentences"],
                "pred_sentences": parsed.get("incidental_sentences", []) if parsed else None,
                "parse_failed": parsed is None,
            })

    return results


class MacroF1EarlyStoppingCallback(TrainerCallback):
    def __init__(self, eval_records, tokenizer, run_name, eval_steps=EVAL_STEPS, patience=3):
        self.eval_records = eval_records
        self.tokenizer = tokenizer
        self.run_name = run_name
        self.eval_steps = eval_steps
        self.patience = patience
        self.best_f1 = -1.0
        self.no_improve = 0
        self.best_step = 0
        self.best_metrics = None
        self.best_ckpt_dir = f"./qlora_best_{run_name}"

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.eval_steps != 0 or state.global_step == 0:
            return control

        model = kwargs["model"]
        results = generate_predictions(model, self.tokenizer, self.eval_records, n=100)
        metrics = run_evaluation_fuzzy(results, label=f"Step {state.global_step}")

        wandb.log({
            "val_macro_f1": metrics["macro_f1"],
            "val_micro_f1": metrics["micro_f1"],
            "val_weighted_f1": metrics["weighted_f1"],
            "val_avg_neg": metrics["avg_neg"],
            "val_avg_pos": metrics["avg_pos"],
            "val_parse_failures": metrics["parse_failures"],
            "step": state.global_step,
        })

        if metrics["macro_f1"] > self.best_f1:
            self.best_f1 = metrics["macro_f1"]
            self.best_step = state.global_step
            self.best_metrics = metrics
            self.no_improve = 0
            model.save_pretrained(self.best_ckpt_dir)
            self.tokenizer.save_pretrained(self.best_ckpt_dir)
            print(f"  New best saved -> {self.best_ckpt_dir}")
        else:
            self.no_improve += 1
            print(f"  No improvement ({self.no_improve}/{self.patience})")

        if self.no_improve >= self.patience:
            print(f"\nEarly stopping at step {state.global_step}. Best step {self.best_step}, macro F1={self.best_f1:.4f}")
            control.should_training_stop = True

        return control


print("All eval helpers + modality-specific build_system_prompt loaded.")

def apply_memory_override(raw_output: dict, memory: SentenceMemory) -> dict:
    """Post-process an LLM extraction, overriding any sentence that matches memory."""
    final_sentences = []
    for s in raw_output.get("incidental_sentences", []):
        hit = memory.lookup(s)
        if hit is not None:
            if hit["is_IF"] and hit["corrected_sentence"]:
                final_sentences.append(hit["corrected_sentence"])
            # else: memory says this should be excluded -- drop it
        else:
            final_sentences.append(s)
    return {"contains_IF": len(final_sentences) > 0, "incidental_sentences": final_sentences}

WANDB_API_KEY not set -- running with wandb disabled.
Indexed 41 ACR findings for modality=thoracic (12,750 chars condensed)
All eval helpers + modality-specific build_system_prompt loaded.


In [80]:
# ===========================================================================
# 8. Main
# ===========================================================================

if __name__ == "__main__":
    paths = PATHS[MODALITY]

    print(f"Loading {MODALITY} model...")
    model, tokenizer = load_merged_model(paths["kd_base"], paths["qlora_adapter"], paths["merged_dir"])
    # Requires build_messages, build_system_prompt, parse_output, generate_predictions
    # (and EVAL_BATCH_SIZE) to already be defined in this notebook -- from your training pipeline.

    print(f"Loading {MODALITY} records...")
    records = load_records_for_modality(MODALITY, split="train")
    print(f"Loaded {len(records)} records")

    print("Loading sentence embedding backend...")
    embed_backend = BertStyleEmbedding()

    print("Building pairs + standalone selections...")
    pairs, standalone = build_pairs_and_standalone(records, embed_backend)
    print(f"Built {len(pairs)} pairs, {len(standalone)} standalone cases")

    # Batch-generate raw LLM extractions ONCE for every report_b + standalone report,
    # then apply memory override per-case afterward -- avoids one .generate() call per case.
    needed_ids = {p["report_b"] for p in pairs} | {r["report_id"] for r in standalone}
    needed_records = [r for r in records if r["report_id"] in needed_ids]
    print(f"Batch-generating raw predictions for {len(needed_records)} reports...")
    raw_preds = run_batched_raw_predictions(model, tokenizer, needed_records)

    pair_results = run_pair_generalization_test(pairs, raw_preds, embed_backend)
    summarize(pair_results, "TEST 1: pair generalization")

    self_results = run_self_consistency_test(standalone, raw_preds, embed_backend)
    summarize(self_results, "TEST 2: self-consistency")

    # Save raw results for manual inspection
    with open(f"/kaggle/working/{MODALITY}_pair_test_results.json", "w") as f:
        json.dump(pair_results, f, indent=2)
    with open(f"/kaggle/working/{MODALITY}_self_consistency_results.json", "w") as f:
        json.dump(self_results, f, indent=2)

Loading thoracic model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading thoracic records...
Loaded 1000 records
Loading sentence embedding backend...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building pairs + standalone selections...
  bin 0.98-1.00: found 33 pairs (target 33)
  bin 0.95-0.98: found 33 pairs (target 33)
  bin 0.90-0.95: found 33 pairs (target 33)
Built 99 pairs, 100 standalone cases
Batch-generating raw predictions for 199 reports...

=== TEST 1: pair generalization ===
n_cases: 99   passed: 92   pass_rate: 92.93%
failures -- extraction_miss: 6   memory_mismatch: 1

=== TEST 2: self-consistency ===
n_cases: 100   passed: 99   pass_rate: 99.00%
failures -- extraction_miss: 1   memory_mismatch: 0


In [81]:
# ===========================================================================
# 9. Metric 1 -- Main results table (current modality only)
# ===========================================================================
main_table_rows = []

n = len(pair_results)
rate = sum(r["pass"] for r in pair_results) / n if n else float("nan")
main_table_rows.append((MODALITY.capitalize(), "Generalization", n, rate))

n = len(self_results)
rate = sum(r["pass"] for r in self_results) / n if n else float("nan")
main_table_rows.append((MODALITY.capitalize(), "Consistency", n, rate))

print(f"{'Modality':<10}{'Test':<15}{'n':>6}{'Pass rate':>12}")
for mod, test, n, rate in main_table_rows:
    print(f"{mod:<10}{test:<15}{n:>6}{rate:>11.1%}")

Modality  Test                n   Pass rate
Thoracic  Generalization     99      92.9%
Thoracic  Consistency       100      99.0%


In [82]:
# ===========================================================================
# 10. Metric 2 -- Generalization difficulty (pass rate by similarity bin, current modality only)
# ===========================================================================
SIM_BINS = [(0.98, 1.001), (0.95, 0.98), (0.90, 0.95)]

def bin_pass_rates(pair_res):
    rows = []
    for lo, hi in SIM_BINS:
        in_bin = [r for r in pair_res if lo <= r["similarity_a_b"] < hi]
        n = len(in_bin)
        rate = sum(r["pass"] for r in in_bin) / n if n else float("nan")
        rows.append((f"{lo:.2f}-{min(hi,1.00):.2f}", n, rate))
    return rows

print(f"\n=== Generalization difficulty -- {MODALITY} ===")
print(f"{'Similarity range':<20}{'n pairs':>10}{'Pass rate':>12}")
for rng, n, rate in bin_pass_rates(pair_results):
    print(f"{rng:<20}{n:>10}{rate:>11.1%}")


=== Generalization difficulty -- thoracic ===
Similarity range       n pairs   Pass rate
0.98-1.00                   33      90.9%
0.95-0.98                   33      93.9%
0.90-0.95                   33      93.9%


In [83]:
# ===========================================================================
# 11. Metric 3 -- Threshold sensitivity (current modality only)
# ===========================================================================
THRESHOLDS_TO_TEST = [0.88, 0.90, 0.92, 0.94]

def pass_rate_at_threshold(pairs, raw_preds, embed_backend, threshold):
    outcomes = []
    for pair in pairs:
        memory = SentenceMemory(backend=embed_backend, sim_threshold=threshold)
        modified = default_modify_sentence(pair["sentence_a"])
        memory.add_correction(trigger_sentence=pair["sentence_a"], corrected_sentence=modified, is_if=True)

        raw_sentences_b = raw_preds.get(pair["report_b"], [])
        final_sents = apply_memory_override({"incidental_sentences": raw_sentences_b}, memory)["incidental_sentences"]

        suppressed_original = (
            not exact_in_list(pair["sentence_b"], final_sents)
            and not exact_in_list(pair["sentence_a"], final_sents)
        )
        injected_modification = sentence_in_list(modified, final_sents, 0.85)
        outcomes.append(suppressed_original and injected_modification)
    return sum(outcomes) / len(outcomes) if outcomes else float("nan")

threshold_results = {t: pass_rate_at_threshold(pairs, raw_preds, embed_backend, t) for t in THRESHOLDS_TO_TEST}

print(f"{'Sim threshold':<15}{'Gen. pass rate (' + MODALITY + ')':>28}")
for t, rate in threshold_results.items():
    print(f"{t:<15}{rate:>27.1%}")

Sim threshold     Gen. pass rate (thoracic)
0.88                                 92.9%
0.9                                  92.9%
0.92                                 90.9%
0.94                                 90.9%
